In [ ]:
from jointfm_client import bootstrap_notebook
bootstrap_notebook(add_src_root=True)

# Sample Forecast
Forecast USD portfolio NAV and risk from 100 positive float daily observations: equity index level, 10-year Treasury yield, and one EUR/USD FX rate. Request sample-path forecasts for portfolio NAV and realized volatility over the next 10 ordinal horizons.

In [ ]:
from pathlib import Path

import pandas as pd

from jointfm_client import ColumnSpec, JointFMClient

HISTORY_PATH = Path('notebooks/history.csv')
FEATURE_COLUMNS = ['equity_index_level', 'treasury_10y_yield', 'eur_usd_rate']
TARGET_COLUMNS = ['portfolio_nav', 'realized_volatility']
N_SAMPLES = 10
INPUT_STEPS = 100
OUTPUT_HORIZONS = 2
EXPECTED_COLUMNS = FEATURE_COLUMNS + TARGET_COLUMNS
QUERY_TIMES = list(range(INPUT_STEPS, INPUT_STEPS + OUTPUT_HORIZONS))

history = pd.read_csv(HISTORY_PATH, dtype=float)
if list(history.columns) != EXPECTED_COLUMNS:
    raise ValueError(f'Expected columns {EXPECTED_COLUMNS!r}, got {list(history.columns)!r}')
if len(history) != INPUT_STEPS:
    raise ValueError(f'Expected {INPUT_STEPS} history rows, got {len(history)}')

columns = [
    *[
        ColumnSpec(name=column_name, modality='numeric', role='feature')
        for column_name in FEATURE_COLUMNS
    ],
    *[
        ColumnSpec(name=column_name, modality='numeric', role='target')
        for column_name in TARGET_COLUMNS
    ],
]
client = JointFMClient.from_env()
result = client.forecast_samples(
    history,
    query_times=QUERY_TIMES,
    requested_columns=TARGET_COLUMNS,
    columns=columns,
    n_samples=N_SAMPLES,
    seed=7,
)
forecast = result.to_pandas_tidy()
expected_forecast_rows = OUTPUT_HORIZONS * len(TARGET_COLUMNS) * N_SAMPLES
if len(forecast) != expected_forecast_rows:
    raise ValueError(f'Expected {expected_forecast_rows} forecast rows, got {len(forecast)}')
forecast